In [ ]:
# 🧩 Scenario
# A company receives 1000+ resumes daily and wants to automate candidate screening.

# They need a system that:

# Extracts candidate details

# Structures the data

# Tags each processed document with Student ID (who processed it)

# 🎯 Your Task
# Build a document processing system using Azure.

# ⚙️ Requirements
# You MUST use:
# Azure Document Intelligence
# Azure Cognitive Services
# 🔧 Functional Requirements
# Upload resume (PDF/Image)

# Extract:
# Name
# Skills
# Email
# Attach Student ID to output

# Use:
# API Key
# Endpoint

# 📥 Input
# Resume file
# Student ID

In [2]:
!pip install azure-ai-documentintelligence azure-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 9.8 MB/s eta 0:00:00


In [3]:

# 📌 Resume Processing System using Azure

import re
import json
from datetime import datetime
from google.colab import userdata
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient


# 🔐 CONFIGURATION

ENDPOINT = "https://mrinaldocintelligence.cognitiveservices.azure.com/"
api_key = userdata.get("DOCUMENT_INTELLIGENCE_ENDPOINT")

# 🚀 Initialize Azure Client

client = DocumentIntelligenceClient(
    endpoint=ENDPOINT,
    credential=AzureKeyCredential(api_key)
)

# 🧠 Function: Extract Text from Resume

def extract_text(file_path):
    with open(file_path, "rb") as f:
        poller = client.begin_analyze_document(
            model_id="prebuilt-read",
            body=f
        )

    result = poller.result()

    text = ""
    for page in result.pages:
        for line in page.lines:
            text += line.content + "\n"

    return text.strip()

# 📧 Function: Extract Email

def extract_email(text):
    match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    return match.group(0) if match else None

# 👤 Function: Extract Name
def extract_name(text):
    lines = text.split("\n")

    for line in lines:
        line = line.strip()

        # Skip unwanted lines
        if not line:
            continue
        if "@" in line:
            continue
        if re.search(r'\d', line):
            continue

        words = line.split()


        if 2 <= len(words) <= 4:
            return line

    return None

# 🛠️ Function: Extract Skills
def extract_skills(text):
    skills_db = [
        "Python", "Java", "C", "C++", "SQL", "Azure", "AWS",
        "Machine Learning", "Deep Learning", "Data Analysis",
        "HTML", "CSS", "JavaScript", "React", "Node.js",
        "TensorFlow", "Pandas", "NumPy"
    ]

    found_skills = []

    for skill in skills_db:
        if skill.lower() in text.lower():
            found_skills.append(skill)

    return list(set(found_skills))

#  Main Function

def process_resume(file_path, student_id):

    print("\n🔍 Processing Resume...")

    # Step 1: Extract text
    text = extract_text(file_path)

    # Step 2: Extract fields
    name = extract_name(text)
    email = extract_email(text)
    skills = extract_skills(text)

    # Step 3: Structure output
    result = {
        "student_id": student_id,
        "name": name,
        "skills": skills,
        "email": email
    }

    # Step 4: Save log (optional)
    log = {
        "timestamp": str(datetime.now()),
        "data": result
    }

    with open("resume_logs.json", "a") as f:
        f.write(json.dumps(log) + "\n")

    return result



# ▶ RUN PROGRAM
if __name__ == "__main__":

    # 📥 Input
    file_path = "/content/MrinaLResume (2).pdf"
    student_id = "BTECH2026_205"

    # 🚀 Process
    output = process_resume(file_path, student_id)

    # 📤 Output
    print("\n Extracted Data:")
    print(json.dumps(output, indent=2))


🔍 Processing Resume...

 Extracted Data:
{
  "student_id": "BTECH2026_205",
  "name": "MRINAL MAYANK",
  "skills": [
    "SQL",
    "AWS",
    "React",
    "CSS",
    "Java",
    "HTML",
    "C++",
    "Pandas",
    "Node.js",
    "Python",
    "C",
    "JavaScript"
  ],
  "email": "mrinalmayank20@gmail.com"
}
